# Three levels of tokenization, and the trade between them

Character, word, and subword — built by hand, measured against each other, so that byte-pair encoding stops being a name and becomes a solution to a stated problem.

**Runs on:** CPU — about 2 minutes &nbsp;·&nbsp; **Slides:** [Chapter 14 — Text Classification](../../../course-web-slides/ch14/index.html) &nbsp;·&nbsp; **Section:** 01 — Preparing text data

---

## The simplest tokenizer there is

In [ ]:
import re
import string

def standardize(text):
    text = text.lower()
    return "".join(c for c in text if c not in string.punctuation)

def tokenize(text):
    return standardize(text).split()

sentence = "The cat sat on the mat, and the mat was NOT amused."
print(tokenize(sentence))

Lowercase, strip punctuation, split on whitespace. Every decision here is a **loss of information**, and each is defensible in some contexts and not others: lowercasing loses the distinction between *US* and *us*; stripping punctuation loses the question mark.

## Character level

In [ ]:
text = open(__file__).read() if False else (
    "the quick brown fox jumps over the lazy dog. " * 200)

chars = sorted(set(text))
print(f"vocabulary: {len(chars)} characters")
print(f"sequence length for one sentence: "
      f"{len('the quick brown fox jumps over the lazy dog.')}")

**Tiny vocabulary, very long sequences.** Nothing is ever out-of-vocabulary — it can encode any string in any language — and the model has to learn spelling before it can learn meaning.

## Word level

In [ ]:
from keras.datasets import imdb

word_index = imdb.get_word_index()
print(f"vocabulary: {len(word_index):,} words")

# The long tail, which is where the trouble is.
import numpy as np
ranks = np.arange(1, len(word_index) + 1)
covered = {10_000: 0, 20_000: 0, 50_000: 0}
print("\nTruncating the vocabulary drops the rare words:")
for k in covered:
    print(f"  keep the {k:,} most common -> "
          f"{len(word_index) - k:,} words become <UNK>")

**Short sequences, huge vocabulary.** And the vocabulary is a long tail: keeping the top 10,000 words is standard and throws away 80,000 of them. Every discarded word becomes `<UNK>` — identical to every other discarded word.

## Byte-pair encoding, built

The idea: start from characters, then repeatedly **merge the most frequent adjacent pair** into a new token. Common words end up as one token; rare words decompose into pieces.

In [ ]:
from collections import Counter

def get_stats(vocab):
    pairs = Counter()
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[symbols[i], symbols[i + 1]] += freq
    return pairs

def merge_vocab(pair, vocab):
    out = {}
    bigram = re.escape(" ".join(pair))
    p = re.compile(r"(?<!\S)" + bigram + r"(?!\S)")
    for word in vocab:
        out[p.sub("".join(pair), word)] = vocab[word]
    return out

corpus = ("low low low low low lowest lowest newer newer newer newer "
          "newer newer wider wider wider new new")
vocab = Counter(corpus.split())
vocab = {" ".join(w) + " </w>": c for w, c in vocab.items()}

print("start:", list(vocab)[:3], "...\n")
merges = []
for i in range(10):
    pairs = get_stats(vocab)
    if not pairs:
        break
    best = max(pairs, key=pairs.get)
    merges.append(best)
    vocab = merge_vocab(best, vocab)
    print(f"merge {i+1:2d}: {best[0]!r} + {best[1]!r} -> {''.join(best)!r}"
          f"   (seen {pairs[best]} times)")

In [ ]:
print("\nfinal vocabulary:")
for w, c in vocab.items():
    print(f"  {w!r}  x{c}")

`er</w>`, `low`, `new` emerge as units because they are frequent. **A word never seen before still encodes** — into pieces — which is what removes `<UNK>` from the picture entirely.

## The trade, measured

In [ ]:
sample = "internationalization pretokenization antidisestablishmentarianism"

print(f"{'scheme':12s} {'vocabulary':>12s} {'tokens for the sample':>24s}")
print("-" * 50)
print(f"{'character':12s} {len(set(sample)):>12,} {len(sample):>24}")
print(f"{'word':12s} {88584:>12,} {len(sample.split()):>24}")
print(f"{'subword':12s} {32000:>12,} {'~12 (estimated)':>24}")

Subword sits between the two on **both** axes, which is why every model in chapters 15 through 17 uses it. It is not a compromise so much as a solution: short sequences *and* a small vocabulary *and* no unknown words.

## TextVectorization

In [ ]:
import keras
from keras import layers

text_vectorization = layers.TextVectorization(
    output_mode="int",
    max_tokens=20,
    output_sequence_length=10,
)

dataset = ["I write, erase, rewrite",
           "Erase again, and then",
           "A poppy blooms."]
text_vectorization.adapt(dataset)
print(text_vectorization.get_vocabulary())

In [ ]:
encoded = text_vectorization("I write, rewrite, and still rewrite again")
print("encoded:", encoded.numpy())

vocab = text_vectorization.get_vocabulary()
inverse = dict(enumerate(vocab))
print("decoded:", " ".join(inverse[int(i)] for i in encoded if int(i) != 0))

> ⚠️ **`adapt()` is fitting.** It learns the vocabulary from data, so it must see the training split only. Calling it on the full dataset puts test vocabulary into the model — chapter 5's leak, wearing a Keras layer.

## Index 0 and index 1 are reserved

In [ ]:
print(f"index 0: {vocab[0]!r}  (padding)")
print(f"index 1: {vocab[1]!r}  (out of vocabulary)")
print()
print("Sequences are padded to a fixed length with 0, and every unknown")
print("word collapses to 1. Both are information the model has to work")
print("around, and both are visible in the encoded output above.")

---

## What to take away

- Character level: tiny vocabulary, very long sequences, no unknown words.
- Word level: short sequences, enormous vocabulary, a long tail that becomes `<UNK>`.
- **BPE merges frequent adjacent pairs**, getting both short sequences and a small vocabulary.
- `adapt()` fits on data — training split only.